In [0]:
# ============================================================
# Avaliação de Qualidade de Dados — silver.aerodromos
# ============================================================

from pyspark.sql.functions import col, count, when, isnull, isnan, trim, length, approx_count_distinct

TABELA = "voe_bem.silver.aerodromos"
df = spark.table(TABELA)

print(f"📋 Tabela: {TABELA}")
print(f"📊 Total de registros: {df.count():,}")
print(f"🗂️  Total de colunas: {len(df.columns)}")
print()

# ----------------------------------------------------------
# 1. Esquema / metadados
# ----------------------------------------------------------
print("=" * 80)
print("1. ESQUEMA DA TABELA")
print("=" * 80)
df.printSchema()

# ----------------------------------------------------------
# 2. Amostra de dados
# ----------------------------------------------------------
print("=" * 80)
print("2. AMOSTRA (10 primeiras linhas)")
print("=" * 80)
display(df.limit(10))

# ----------------------------------------------------------
# 3. Perfil de nulos por coluna
# ----------------------------------------------------------
print("=" * 80)
print("3. PERFIL DE NULOS / VAZIOS POR COLUNA")
print("=" * 80)

colunas = df.columns
exprs_nulos = []
exprs_vazios = []
for c in colunas:
    exprs_nulos.append(count(when(col(c).isNull(), c)).alias(c))
    # trim para strings detectar vazios/whitespace
    exprs_vazios.append(count(when(col(c).rlike("^\\s*$") | (trim(col(c)) == ""), c)).alias(c))

nulos_df = df.agg(*exprs_nulos)
vazios_df = df.agg(*exprs_vazios)

nulos_row = nulos_df.collect()[0]
vazios_row = vazios_df.collect()[0]

total = df.count()
print(f"{'Coluna':<40} {'Nulos':>10} {'%Nulos':>10} {'Vazios/Ws':>10} {'%Vazios':>10}")
print("-" * 80)
for c in colunas:
    n = nulos_row[c]
    v = vazios_row[c]
    pn = (n / total * 100) if total else 0
    pv = (v / total * 100) if total else 0
    print(f"{c:<40} {n:>10,} {pn:>9.2f}% {v:>10,} {pv:>9.2f}%")

# ----------------------------------------------------------
# 4. Cardinalidade (distinct) por coluna
# ----------------------------------------------------------
print("=" * 80)
print("4. CARDINALIDADE (DISTINCT APROXIMADO) POR COLUNA")
print("=" * 80)
exprs_distinct = [approx_count_distinct(col(c)).alias(c) for c in colunas]
distinct_row = df.agg(*exprs_distinct).collect()[0]
print(f"{'Coluna':<40} {'Distinct':>12} {'%Unique':>10}")
print("-" * 62)
for c in colunas:
    d = distinct_row[c]
    pu = (d / total * 100) if total else 0
    print(f"{c:<40} {d:>12,} {pu:>9.2f}%")

# ----------------------------------------------------------
# 5. Duplicatas
# ----------------------------------------------------------
print("=" * 80)
print("5. ANÁLISE DE DUPLICATAS")
print("=" * 80)
total_distinct = df.distinct().count()
dupl = total - total_distinct
print(f"Linhas totais:            {total:,}")
print(f"Linhas distintas:         {total_distinct:,}")
print(f"Linhas duplicadas:        {dupl:,} ({dupl/total*100:.2f}%)")

# ----------------------------------------------------------
# 6. Estatísticas descritivas (numéricas e string)
# ----------------------------------------------------------
print("=" * 80)
print("6. ESTATÍSTICAS DESCRITIVAS")
print("=" * 80)
display(df.summary("count", "min", "25%", "50%", "75%", "max"))

# ----------------------------------------------------------
# 7. Resumo e recomendações de qualidade
# ----------------------------------------------------------
print("=" * 80)
print("7. RECOMENDAÇÕES DE TRATAMENTO DE QUALIDADE")
print("=" * 80)

print("""
Com base no perfil acima, verifique e aplique os seguintes tratamentos:

A) NULIDADE
   - Colunas com > 50% de nulos: avaliar remoção ou reestruturar fonte.
   - Colunas críticas (chaves, códigos, nomes): aplicar coalesce com valor padrão ou rejeitar registro.
   - Para campos opcionais, documentar a regra de nulidade.

B) STRINGS VAZIAS / WHITESPACE
   - Substituir strings vazias por NULL (trim + nullif) para padronizar.
   - Aplicar trim() em todas as colunas de texto.
   - Padronizar case (UPPER/LOWER) conforme regra de negócio.

C) DUPLICATAS
   - Se houver duplicatas, definir coluna(s) de chave natural e aplicar dropDuplicates().
   - Investigar origem da duplicação (reprocessamento, falta de dedup na camada bronze).

D) CARDINALIDADE
   - Colunas com baixa cardinalidade: validar se são categóricas e aplicar restrições CHECK ou dicionário de valores.
   - Colunas com 100% unique: confirmar se são chaves primárias esperadas.

E) PADRONIZAÇÃO
   - Códigos de aeródromo (ICAO/IATA): validar formato (4 letras ICAO, 3 letras IATA).
   - Coordenadas: validar range de latitude [-90, 90] e longitude [-180, 180].
   - Datas: garantir formato ISO 8601 e validar intervalo plausível.

F) GOVERNANÇA
   - Aplicar tags Unity Catalog (pii, quality_tier=silver) na tabela e colunas.
 - Adicionar comentários/descrição nas colunas via ALTER TABLE ... COLUMN COMMENT.
   - Criar expectation checks (DLT) ou CHECK constraints onde aplicável.
 - Considerar criação de metric view para monitoramento contínuo de KPIs de qualidade.

G) MONITORAMENTO
   - Implementar verificações recorrentes via DLT expectations ou dbt tests.
   - Logar métricas de qualidade (nulidade, duplicidade) por execução.
""")
print("\n✅ Avaliação concluída. Revise os pontos acima e priorize tratamentos conforme criticidade.")

In [0]:
# ============================================================
# Validações de Contexto de Negócio — silver.aerodromos
# ============================================================
# Regras validadas:
#   1. ICAO: 4 caracteres alfanuméricos, prefixo brasileiro (S + [A-Z])
#   2. CIAD: 2 letras (sigla UF) + 4 dígitos
#   3. uf_nome / uf_servido_nome: pertence ao conjunto das 27 UFs válidas
#   4. latitude_dms: formato DMS válido, 0–90°, hemisfério N ou S
#   5. longitude_dms: formato DMS válido, 0–180°, hemisfério W (Brasil)
#   6. altitude_m: intervalo plausível [0, 3000] m
#   7. situacao: domínio {Cadastrado, Interditado}
#   8. Colunas críticas não nulas (icao, ciad, nome, latitude_dms, longitude_dms)
#   9. ICAO é chave única (primary key esperada)
#  10. Consistência: municipio e uf_nome ambos nulos ou ambos preenchidos
#  11. Consistência: municipio_servido e uf_servido_nome ambos nulos ou ambos preenchidos
#  12. uf_nome e uf_servido_nome coerentes quando município = município servido
# ============================================================

from pyspark.sql.functions import col, count, when, trim, upper, length, regexp_extract, rlike

TABELA = "voe_bem.silver.aerodromos"
df = spark.table(TABELA)
total = df.count()

UFS_VALIDAS = [
    "Acre", "Alagoas", "Amapá", "Amazonas", "Bahia", "Ceará",
    "Distrito Federal", "Espírito Santo", "Goiás", "Maranhão",
    "Mato Grosso", "Mato Grosso do Sul", "Minas Gerais", "Pará",
    "Paraíba", "Paraná", "Pernambuco", "Piauí", "Rio de Janeiro",
    "Rio Grande do Norte", "Rio Grande do Sul", "Rondônia", "Roraima",
    "Santa Catarina", "São Paulo", "Sergipe", "Tocantins",
]
SITUACOES_VALIDAS = ["Cadastrado", "Interditado"]

violacoes = []  # lista de (regra, descrição, total, percentual)

# ----------------------------------------------------------
# 1. ICAO — formato 4 letras maiúsculas + prefixo brasileiro
# ----------------------------------------------------------
regra = "1. ICAO — formato e prefixo"
# Prefixos brasileiros válidos: SB, SN, SS, SD, SW, SJ, SI
v = df.filter(~col("icao").rlike("^[A-Z0-9]{4}$") | ~col("icao").rlike("^S[BNSWSDJI]")).count()
if v:
    violacoes.append((regra, "ICAO fora do padrão 4 alfanuméricos ou prefixo brasileiro (SB/SN/SS/SD/SW/SJ/SI)", v, v / total * 100))
print(f"✓ {regra}: {total - v} OK, {v} violações")

# ----------------------------------------------------------
# 2. CIAD — 2 letras + 4 dígitos
# ----------------------------------------------------------
regra = "2. CIAD — formato"
v = df.filter(~col("ciad").rlike(r"^[A-Z]{2}\d{4}$")).count()
if v:
    violacoes.append((regra, "CIAD fora do padrão 2 letras + 4 dígitos", v, v / total * 100))
print(f"✓ {regra}: {total - v} OK, {v} violações")

# ----------------------------------------------------------
# 3. uf_nome e uf_servido_nome — domínio válido
# ----------------------------------------------------------
for coluna in ["uf_nome", "uf_servido_nome"]:
    regra = f"3. {coluna} — domínio UFs válidas"
    v = df.filter(col(coluna).isNotNull() & ~col(coluna).isin(UFS_VALIDAS)).count()
    if v:
        violacoes.append((regra, f"{coluna} com valor fora das 27 UFs brasileiras", v, v / total * 100))
    print(f"✓ {regra}: {total - v} OK, {v} violações")

# ----------------------------------------------------------
# 4. latitude_dms — formato e intervalo
# ----------------------------------------------------------
regra = "4. latitude_dms — formato e intervalo"
# Padrão esperado: NN°NN'NN"N|S  (graus 00–90)
lat_regex = r"^(\d{2})°(\d{2})'(\d{2})\"([NS])$"
v_fmt = df.filter(~col("latitude_dms").rlike(lat_regex)).count()
if v_fmt:
    violacoes.append((regra, f"latitude_dms fora do formato DMS esperado", v_fmt, v_fmt / total * 100))

# Extrair graus e validar intervalo (apenas para os que passaram o regex)
df_lat_ok = df.filter(col("latitude_dms").rlike(lat_regex))
v_range = df_lat_ok.filter(regexp_extract(col("latitude_dms"), lat_regex, 1).cast("int") > 90).count()
if v_range:
    violacoes.append((regra, f"latitude_dms com graus > 90", v_range, v_range / total * 100))
v = v_fmt + v_range
print(f"✓ {regra}: {total - v} OK, {v} violações (formato: {v_fmt}, intervalo: {v_range})")

# ----------------------------------------------------------
# 5. longitude_dms — formato, intervalo e hemisfério W
# ----------------------------------------------------------
regra = "5. longitude_dms — formato, intervalo e hemisfério"
lon_regex = r"^(\d{3})°(\d{2})'(\d{2})\"([EW])$"
v_fmt = df.filter(~col("longitude_dms").rlike(lon_regex)).count()
if v_fmt:
    violacoes.append((regra, f"longitude_dms fora do formato DMS esperado", v_fmt, v_fmt / total * 100))

df_lon_ok = df.filter(col("longitude_dms").rlike(lon_regex))
v_range = df_lon_ok.filter(regexp_extract(col("longitude_dms"), lon_regex, 1).cast("int") > 180).count()
if v_range:
    violacoes.append((regra, f"longitude_dms com graus > 180", v_range, v_range / total * 100))

# Brasil está inteiramente no hemisfério Oeste
v_hemi = df_lon_ok.filter(regexp_extract(col("longitude_dms"), lon_regex, 4) != "W").count()
if v_hemi:
    violacoes.append((regra, f"longitude_dms com hemisfério diferente de W (Brasil)", v_hemi, v_hemi / total * 100))

v = v_fmt + v_range + v_hemi
print(f"✓ {regra}: {total - v} OK, {v} violações (formato: {v_fmt}, intervalo: {v_range}, hemisfério: {v_hemi})")

# ----------------------------------------------------------
# 6. altitude_m — intervalo plausível
# ----------------------------------------------------------
regra = "6. altitude_m — intervalo [0, 3000]"
v = df.filter((col("altitude_m") < 0) | (col("altitude_m") > 3000)).count()
if v:
    violacoes.append((regra, "altitude_m fora do intervalo plausível [0, 3000] m", v, v / total * 100))
print(f"✓ {regra}: {total - v} OK, {v} violações")

# ----------------------------------------------------------
# 7. situacao — domínio válido
# ----------------------------------------------------------
regra = "7. situacao — domínio {Cadastrado, Interditado}"
v = df.filter(~col("situacao").isin(SITUACOES_VALIDAS)).count()
if v:
    violacoes.append((regra, f"situacao com valor fora do domínio {SITUACOES_VALIDAS}", v, v / total * 100))
print(f"✓ {regra}: {total - v} OK, {v} violações")

# ----------------------------------------------------------
# 8. Colunas críticas não nulas
# ----------------------------------------------------------
regra = "8. Colunas críticas não nulas"
col_criticas = ["icao", "ciad", "nome", "latitude_dms", "longitude_dms"]
v_crit = 0
for c in col_criticas:
    n = df.filter(col(c).isNull()).count()
    if n:
        violacoes.append((regra, f"Coluna crítica '{c}' com {n} nulos", n, n / total * 100))
        v_crit += n
print(f"✓ {regra}: {v_crit} nulos totais em {len(col_criticas)} colunas")

# ----------------------------------------------------------
# 9. ICAO — unicidade (chave primária esperada)
# ----------------------------------------------------------
regra = "9. ICAO — unicidade (primary key)"
dup_icao = df.groupBy("icao").count().filter("count > 1")
v = dup_icao.count()
if v:
    total_dup = dup_icao.agg({"count": "sum"}).collect()[0][0] - v
    violacoes.append((regra, f"ICAO duplicado: {v} códigos com mais de um registro", total_dup, total_dup / total * 100))
print(f"✓ {regra}: {total - v} códigos únicos, {v} códigos duplicados")
if v:
    print("  Duplicados de ICAO:")
    dup_icao.orderBy("count", ascending=False).show(20, truncate=False)

# ----------------------------------------------------------
# 10. Consistência: municipio e uf_nome ambos nulos ou ambos preenchidos
# ----------------------------------------------------------
regra = "10. Consistência municipio ↔ uf_nome"
v = df.filter(
    (col("municipio").isNull() & col("uf_nome").isNotNull()) |
    (col("municipio").isNotNull() & col("uf_nome").isNull())
).count()
if v:
    violacoes.append((regra, "municipio e uf_nome com nulidade inconsistente", v, v / total * 100))
print(f"✓ {regra}: {total - v} consistentes, {v} inconsistentes")

# ----------------------------------------------------------
# 11. Consistência: municipio_servido e uf_servido_nome
# ----------------------------------------------------------
regra = "11. Consistência municipio_servido ↔ uf_servido_nome"
v = df.filter(
    (col("municipio_servido").isNull() & col("uf_servido_nome").isNotNull()) |
    (col("municipio_servido").isNotNull() & col("uf_servido_nome").isNull())
).count()
if v:
    violacoes.append((regra, "municipio_servido e uf_servido_nome com nulidade inconsistente", v, v / total * 100))
print(f"✓ {regra}: {total - v} consistentes, {v} inconsistentes")

# ----------------------------------------------------------
# 12. Coerência: quando municipio = municipio_servido, uf_nome deve = uf_servido_nome
# ----------------------------------------------------------
regra = "12. Coerência uf_nome = uf_servido_nome (mesmo município)"
v = df.filter(
    (col("municipio").isNotNull()) &
    (col("municipio") == col("municipio_servido")) &
    (col("uf_nome") != col("uf_servido_nome"))
).count()
if v:
    violacoes.append((regra, "municipio = municipio_servido mas uf_nome ≠ uf_servido_nome", v, v / total * 100))
print(f"✓ {regra}: {total - v} coerentes, {v} incoerentes")

# ============================================================
# Resumo final
# ============================================================
print()
print("=" * 80)
print("RESUMO DE VALIDAÇÕES DE NEGÓCIO")
print("=" * 80)
print(f"Total de registros avaliados: {total:,}")
print(f"Total de regras validadas: 12")
print(f"Regras com violações: {len(violacoes)}")
print()

if violacoes:
    print(f"{'Regra':<55} {'Descrição':<60} {'Qtd':>6} {'%':>7}")
    print("-" * 130)
    for r, d, q, p in violacoes:
        print(f"{r:<55} {d:<60} {q:>6,} {p:>6.2f}%")
    print()
    print("⚠️  Existem violações que requerem tratamento antes de promover a tabela.")
else:
    print("✅ Todas as validações de negócio passaram sem violações.")

In [0]:
# ============================================================
# Filtro de Qualidade — impedir carga de registros com icao vazio
# ============================================================
# Regra: nenhum registro com icao NULL ou string vazia/whitespace
#         deve chegar à camada silver.
# Estratégia: filtrar na origem (bronze) antes do write para silver.
# ============================================================

from pyspark.sql.functions import col, trim

# ----------------------------------------------------------
# Parâmetros
# ----------------------------------------------------------
TABELA_BRONZE = "voe_bem.bronze.aerodromos"
TABELA_SILVER = "voe_bem.silver.aerodromos"

# ----------------------------------------------------------
# Leitura da camada bronze
# ----------------------------------------------------------
df_bronze = spark.table(TABELA_BRONZE)
total_bronze = df_bronze.count()
print(f"📋 Bronze — total de registros lidos: {total_bronze:,}")

# ----------------------------------------------------------
# Data quality check: icao não pode ser nulo nem vazio
# ----------------------------------------------------------
df_rejeitados = df_bronze.filter(
    col("icao").isNull() | (trim(col("icao")) == "") | col("icao").rlike("^\\s*$")
)
qtd_rejeitados = df_rejeitados.count()
print(f"🚫 Registros rejeitados (icao vazio/nulo): {qtd_rejeitados:,}")

if qtd_rejeitados > 0:
    print("\nAmostra de registros rejeitados:")
    display(df_rejeitados.select("icao", "nome", "ciad").limit(20))

# ----------------------------------------------------------
# Filtrar apenas registros válidos (icao preenchido)
# ----------------------------------------------------------
df_silver = df_bronze.filter(
    col("icao").isNotNull() & (trim(col("icao")) != "") & ~col("icao").rlike("^\\s*$")
)
qtd_validos = df_silver.count()
print(f"✅ Registros válidos para silver: {qtd_validos:,}")
print(f"📈 Taxa de aprovação: {qtd_validos / total_bronze * 100:.2f}%" if total_bronze else "")

# ----------------------------------------------------------
# Escrita na camada silver (overwrite controlado)
# ----------------------------------------------------------
# Usando overwrite para garantir idempotência; ajuste para MERGE
# se precisar de upsert incremental.
df_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_SILVER)

print(f"\n💾 Escrita concluída em {TABELA_SILVER}: {qtd_validos:,} registros")
print("🔒 Garantia de qualidade: nenhum registro com icao vazio foi carregado.")